# Phase 1 matched-order run: XNLI sequential fine-tuning (HI → DE → EN)

This is the Hindi → German → English arm of the matched Phase 1 order comparison. It uses the same no-replay training setup, sample counts, fixed test selection, and seeds as the companion English → Hindi → German notebook; stream order is the intended difference. Results and checkpoints are saved to this order's separate Google Drive folder.

## Before running

1. In Colab, select **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU).
2. Accept the access terms for [`meta-llama/Llama-3.2-1B`](https://huggingface.co/meta-llama/Llama-3.2-1B) and add a Hugging Face read token as a Colab secret named `HF_TOKEN`.
3. Add a **new, rotated** W&B key as a Colab secret named `WANDB_API_KEY`. A key pasted into chat should be revoked and must not be reused. For the CoreWeave-hosted W&B account, also add `WANDB_BASE_URL` with the server URL (for example `https://api.forge.coreweave.com`).
4. Upload `phase1_xnli_multiseed_colab_package.zip` when prompted below. This transfers the source package; training and evaluation run on Colab, not your computer.

This follow-up uses a moderate evaluation (1,000 training and 500 test examples per language) with three training seeds (42, 43, 44). The same fixed test examples are used for every seed; training subsets and model initialization vary by seed. Test accuracy is for reporting only, not for selecting checkpoints or tuning hyperparameters. A successful pilot alone is not evidence that the scientific checkpoint has reproduced robustly.

In [ ]:
%pip -q install "datasets>=3.0" "transformers>=4.46" "peft>=0.13" accelerate bitsandbytes wandb

In [ ]:
import os
import torch
from google.colab import userdata
from huggingface_hub import login

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Set a GPU runtime in Colab, then rerun this cell.")

try:
    hf_token = userdata.get("HF_TOKEN")
    wandb_key = userdata.get("WANDB_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "Add HF_TOKEN and WANDB_API_KEY in Colab's Secrets panel and enable notebook access."
    ) from exc

if not hf_token or not wandb_key:
    raise RuntimeError("Both HF_TOKEN and WANDB_API_KEY Colab secrets are required.")

# Keep credentials in this runtime's environment; never display or save them.
os.environ["WANDB_API_KEY"] = wandb_key
try:
    base_url = userdata.get("WANDB_BASE_URL")
except Exception:
    base_url = None
if base_url:
    os.environ["WANDB_BASE_URL"] = base_url

login(token=hf_token, add_to_git_credential=False)
del hf_token, wandb_key
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("Hugging Face authentication completed; credentials were not printed.")

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import sys

PACKAGE_DIR = Path("/content/continual_learning_package")
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
print("Upload phase1_xnli_multiseed_colab_package.zip")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Upload exactly one component ZIP file.")

archive_path = Path("/content") / zip_names[0]
with ZipFile(archive_path) as archive:
    root = PACKAGE_DIR.resolve()
    for member in archive.infolist():
        destination = (PACKAGE_DIR / member.filename).resolve()
        if destination != root and root not in destination.parents:
            raise ValueError("The uploaded archive contains an unsafe path.")
    archive.extractall(PACKAGE_DIR)

if not (PACKAGE_DIR / "shared" / "continual_learning_lora" / "experiment.py").is_file():
    raise FileNotFoundError("The uploaded ZIP is not the complete continual-learning package.")
sys.path.insert(0, str(PACKAGE_DIR))
print("Reusable experiment package loaded from the uploaded archive.")

In [ ]:
from datasets import load_dataset
from google.colab import drive
from pathlib import Path

SEEDS = [42, 43, 44]
TEST_SPLIT_SEED = 42
LANGUAGES = ["hi", "de", "en"]
TRAIN_SAMPLES_PER_LANGUAGE = 1000
TEST_SAMPLES_PER_LANGUAGE = 500
DATASET_ID = "facebook/xnli"
LABEL_NAMES = {0: "entailment", 1: "neutral", 2: "contradiction"}

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/continual_learning_phase1_hi_de_en")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
from datetime import datetime, timezone
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPERIMENT_GROUP_ID = (
    f"xnli-hi-de-en-multiseed-train{TRAIN_SAMPLES_PER_LANGUAGE}"
    f"-test{TEST_SAMPLES_PER_LANGUAGE}-{RUN_TIMESTAMP}"
)

def format_xnli_row(example):
    label_id = int(example["label"])
    if label_id not in LABEL_NAMES:
        raise ValueError(f"Unexpected XNLI label id: {label_id}")
    prompt = (
        "Task: Natural language inference\n"
        f"Premise: {example['premise']}\n"
        f"Hypothesis: {example['hypothesis']}\n"
        "Choose one label: entailment, neutral, contradiction\n"
        "Answer:"
    )
    return {"prompt": prompt, "target": " " + LABEL_NAMES[label_id]}

train_sets_by_seed = {seed: {} for seed in SEEDS}
test_sets = {}
for language in LANGUAGES:
    train_raw = load_dataset(DATASET_ID, language, split="train")
    test_raw = load_dataset(DATASET_ID, language, split="test")
    if len(test_raw) < TEST_SAMPLES_PER_LANGUAGE:
        raise ValueError(f"Not enough test examples for {language}.")
    test_sets[language] = test_raw.shuffle(seed=TEST_SPLIT_SEED).select(
        range(TEST_SAMPLES_PER_LANGUAGE)
    )
    for seed in SEEDS:
        if len(train_raw) < TRAIN_SAMPLES_PER_LANGUAGE:
            raise ValueError(f"Not enough training examples for {language}.")
        train_subset = train_raw.shuffle(seed=seed).select(
            range(TRAIN_SAMPLES_PER_LANGUAGE)
        )
        train_sets_by_seed[seed][language] = train_subset.map(
            format_xnli_row,
            remove_columns=train_subset.column_names,
            desc=f"Formatting {language} training examples for seed {seed}",
        )
    print(f"{language}: train={train_count} per seed, fixed test={len(test_sets[language])}")

assert all(list(train_sets_by_seed[seed]) == LANGUAGES for seed in SEEDS)
sample_rows = train_sets_by_seed[SEEDS[0]][LANGUAGES[0]].select(range(min(3, len(train_sets_by_seed[SEEDS[0]][LANGUAGES[0]]))))
assert all(set(row) == {"prompt", "target"} for row in sample_rows)
assert len({len(test_sets[language]) for language in LANGUAGES}) == 1
print("Dataset and label formatting checks passed.")

In [ ]:
from shared.continual_learning_lora import LlamaLoRATrainer

MODEL_ID = "meta-llama/Llama-3.2-1B"
print(f"Configured {MODEL_ID} for seeds {SEEDS}; a fresh model will be loaded for each seed.")

In [ ]:
import gc
from transformers import set_seed
from shared.continual_learning_lora import run_sequential_experiment

WANDB_PROJECT = "asymmetric-interference-continual-learning"
seed_results = []
for seed in SEEDS:
    set_seed(seed)
    experiment_id = f"{EXPERIMENT_GROUP_ID}-seed{seed}"
    run_root = DRIVE_ROOT / EXPERIMENT_GROUP_ID / f"seed-{seed}"
    trainer = LlamaLoRATrainer(
        model_name=MODEL_ID,
        output_dir=run_root / "checkpoints",
        task_mode="causal_lm",
        max_length=512,
        learning_rate=2e-4,
        train_batch_size=1,
        eval_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        seed=seed,
        lora_rank=8,
        lora_alpha=16,
        lora_dropout=0.05,
        load_in_4bit=True,
        gradient_checkpointing=True,
        token=True,
    )
    trainer.load_model_and_tokenizer()
    print(f"Starting seed {seed}; trainable parameter summary:")
    trainer.model.print_trainable_parameters()
    result = run_sequential_experiment(
        trainer,
        task_order=LANGUAGES,
        train_datasets=train_sets_by_seed[seed],
        test_datasets=test_sets,
        experiment_id=experiment_id,
        output_dir=run_root / "evaluation",
        dataset_name=DATASET_ID,
        task_name="xnli",
        run_metadata={
            "phase": 1,
            "method": "sequential_finetuning_no_replay",
            "seed": seed,
            "seed_group": EXPERIMENT_GROUP_ID,
            "test_split_seed": TEST_SPLIT_SEED,
            "model": MODEL_ID,
            "train_samples_per_language": TRAIN_SAMPLES_PER_LANGUAGE,
            "test_samples_per_language": TEST_SAMPLES_PER_LANGUAGE,
            "test_split": "test",
            "prompt_protocol": "phase1_b0_nli_prompt_candidate_label_log_likelihood",
        },
        wandb_project=WANDB_PROJECT,
        wandb_mode="online",
    )
    seed_results.append({
        "seed": seed,
        "experiment_id": experiment_id,
        "matrix": result["ledger"].performance_matrix(),
        "metrics": result["metrics"],
        "evaluation_json": str(result["ledger"].json_path),
    })
    print(f"Completed seed {seed}. Evaluation log: {result['ledger'].json_path}")
    del result, trainer
    gc.collect()
    torch.cuda.empty_cache()

print("All seed runs completed.")

In [ ]:
import json
import statistics

matrices = [item["matrix"] for item in seed_results]
mean_matrix = [
    [statistics.mean(matrix[row][column] for matrix in matrices) for column in range(len(LANGUAGES))]
    for row in range(len(LANGUAGES))
]
std_matrix = [
    [statistics.stdev(matrix[row][column] for matrix in matrices) for column in range(len(LANGUAGES))]
    for row in range(len(LANGUAGES))
]
seed_pattern_results = []
for item in seed_results:
    matrix = item["matrix"]
    final_row = matrix[-1]
    diagonal = [matrix[index][index] for index in range(len(LANGUAGES))]
    previous_final_mean = statistics.mean(final_row[:-1])
    previous_learned_mean = statistics.mean(diagonal[:-1])
    last_language_strong = final_row[-1] >= previous_final_mean
    earlier_languages_degraded = previous_final_mean < previous_learned_mean
    seed_pattern_results.append({
        "seed": item["seed"],
        "last_seen_language_strong_relative_to_previous_mean": last_language_strong,
        "earlier_languages_degraded_on_average": earlier_languages_degraded,
        "checkpoint_pattern_passed": last_language_strong and earlier_languages_degraded,
    })
final_mean = mean_matrix[-1]
first_seen_mean = [statistics.mean(matrix[index][index] for matrix in matrices) for index in range(len(LANGUAGES))]

report = {
    "experiment_group_id": EXPERIMENT_GROUP_ID,
    "task_order": LANGUAGES,
    "seeds": SEEDS,
    "train_examples_per_language": TRAIN_SAMPLES_PER_LANGUAGE,
    "test_examples_per_language": TEST_SAMPLES_PER_LANGUAGE,
    "fixed_test_split_seed": TEST_SPLIT_SEED,
    "mean_performance_matrix_rows_after_each_stage": mean_matrix,
    "std_performance_matrix_rows_after_each_stage": std_matrix,
    "mean_final_accuracies_by_language": dict(zip(LANGUAGES, final_mean)),
    "mean_first_seen_accuracies_by_language": dict(zip(LANGUAGES, first_seen_mean)),
    "per_seed_results": seed_results,
    "per_seed_checkpoint_checks": seed_pattern_results,
    "checkpoint_pattern_pass_rate": sum(item["checkpoint_pattern_passed"] for item in seed_pattern_results) / len(seed_pattern_results),
    "checkpoint_pattern_passed_all_seeds": all(item["checkpoint_pattern_passed"] for item in seed_pattern_results),
    "interpretation_note": "Three seeds provide a preliminary repeatability check, not a definitive scientific conclusion.",
}
report_path = DRIVE_ROOT / EXPERIMENT_GROUP_ID / "multiseed_summary.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))
if report["checkpoint_pattern_passed_all_seeds"]:
    print("Checkpoint criteria held for all three seeds. Treat this as preliminary; expand evaluation before a definitive claim.")
else:
    print("Checkpoint criteria did not hold for every seed. Report the per-seed results and mean ± std; do not claim robust reproduction.")
